# Search and A* with an LLM as Engineering Assistant

Everything for this lab in one notebook: the write up, the code, the runs and the tests. Run all cells from top to bottom.

## Task 0: The Search Problem

| Component | Specification |
|---|---|
| State S | A free cell `(row, col)` in the grid |
| Actions A | Up, Down, Left, Right |
| Transition T | `T((r,c), Up) = (r-1, c)` etc., only if the result is not `#` |
| Initial state s0 | The `S` cell, (1, 1) |
| Goal G | The `G` cell, (7, 15) |
| Cost c | 1 per move |

(a) Row and column are enough. Nothing else changes as the robot moves.
(b) An action is invalid if it would move into a `#` cell or off the map.
(c) Yes. Each action has exactly one outcome.
(d) A sequence of valid actions from s0 that ends in G. An optimal solution is one with the fewest moves.

## Task 1: Agent Design (written before prompting)

1. State as a tuple `(r, c)` so it is hashable and can go in sets and dicts.
2. Warehouse as a list of lists of characters.
3. Valid actions: compute the target cell and keep the action if the cell is in bounds and not `#`.
4. Goal test: `state == goal`.
5. Frontier entries: `(f, tie_breaker, state)` in a binary heap, with `g` and `parent` stored in dicts.
6. Path reconstruction: follow `parent` from the goal back to the start and reverse.

Report on termination: found or not, path, path length, states expanded.

## Task 2: Prompt Used

> I am implementing a simple goal-based search agent in Python. The environment is a grid represented by an ASCII map. The agent starts at S and must reach G. The symbols # represent obstacles and . represents free cells. The agent can move up, down, left, or right, and every movement has cost 1. Implement A* search. Use Manhattan distance as the heuristic. The program should represent grid positions as states (row, col) tuples; maintain a heapq frontier ordered by f(n); calculate g(n), h(n) and f(n); keep a closed set so no state is expanded twice; reconstruct the path using parent pointers; report the path, its length and the number of states expanded. Make the heuristic a parameter and add a weight on h so I can experiment. Also write a BFS function with the same report format. Keep it simple and explain the main components.

### Implementation

In [1]:
"""
Lab 2: Search and A* for warehouse robot navigation.

Search problem P = (S, A, T, s0, G, c):
  S  : free grid cells (row, col)
  A  : Up, Down, Left, Right
  T  : move one cell in that direction if the target cell is not '#'
  s0 : the cell marked S
  G  : the cell marked G
  c  : every move costs 1

Contents
  Grid       map loading, valid actions, transition function
  astar()    A* with a pluggable heuristic and an optional weight on h
  bfs()      blind breadth-first search for comparison
  heuristics manhattan, euclidean, zero

"""

import heapq
import math
from collections import deque
from itertools import count

MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


class Grid:
    def __init__(self, text):
        self.cells = [list(line) for line in text.strip("\n").splitlines()]
        self.start = self.goal = None
        for r, row in enumerate(self.cells):
            for c, ch in enumerate(row):
                if ch == "S":
                    self.start = (r, c)
                elif ch == "G":
                    self.goal = (r, c)
        if self.start is None or self.goal is None:
            raise ValueError("Map must contain S and G")

    @classmethod
    def from_file(cls, path):
        with open(path, encoding="utf-8") as f:
            return cls(f.read())

    def passable(self, pos):
        r, c = pos
        return 0 <= r < len(self.cells) and 0 <= c < len(self.cells[r]) and self.cells[r][c] != "#"

    def actions(self, state):
        """Valid actions: those whose result is inside the map and not a wall."""
        return [a for a, (dr, dc) in MOVES.items() if self.passable((state[0] + dr, state[1] + dc))]

    @staticmethod
    def transition(state, action):
        dr, dc = MOVES[action]
        return (state[0] + dr, state[1] + dc)

    def is_goal(self, state):
        return state == self.goal

    def render(self, path):
        cells = [row[:] for row in self.cells]
        for r, c in path[1:-1]:
            cells[r][c] = "*"
        return "\n".join("".join(row) for row in cells)


# Heuristics: h(n) estimates the remaining cost from n to the goal
def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


def euclidean(state, goal):
    return math.hypot(state[0] - goal[0], state[1] - goal[1])


def zero(state, goal):
    return 0


def reconstruct(parent, state):
    path = [state]
    while parent[state] is not None:
        state = parent[state]
        path.append(state)
    return list(reversed(path))


def astar(grid, heuristic=manhattan, weight=1.0, prefer_deeper=False):
    """
    A* search. Frontier is a priority queue ordered by f(n) = g(n) + weight * h(n).

    prefer_deeper: when two nodes have equal f, expand the one with the larger g
    first (it is closer to the goal). This does not change optimality, it only
    changes how ties are broken, which can matter a lot on open grids.

    Returns a dict with found, path, length, expanded.
    """
    start, goal = grid.start, grid.goal
    tie = count()  # tie breaker so heapq never compares states
    g = {start: 0}
    parent = {start: None}
    def key(g_val, h_val):
        f_val = g_val + weight * h_val
        return (f_val, -g_val if prefer_deeper else 0, next(tie))

    frontier = [(*key(0, heuristic(start, goal)), start)]
    closed = set()  # states already expanded
    expanded = 0

    while frontier:
        *_, state = heapq.heappop(frontier)
        if state in closed:  # stale queue entry, a cheaper copy was already expanded
            continue
        closed.add(state)
        expanded += 1
        if grid.is_goal(state):
            path = reconstruct(parent, state)
            return {"found": True, "path": path, "length": len(path) - 1, "expanded": expanded}
        for action in grid.actions(state):
            nxt = grid.transition(state, action)
            new_g = g[state] + 1  # step cost c = 1
            if nxt not in closed and new_g < g.get(nxt, math.inf):
                g[nxt] = new_g
                parent[nxt] = state
                h = heuristic(nxt, goal)
                heapq.heappush(frontier, (*key(new_g, h), nxt))
    return {"found": False, "path": None, "length": None, "expanded": expanded}


def bfs(grid):
    """Blind search: FIFO frontier, no heuristic."""
    start = grid.start
    parent = {start: None}
    frontier = deque([start])
    expanded = 0
    while frontier:
        state = frontier.popleft()
        expanded += 1
        if grid.is_goal(state):
            path = reconstruct(parent, state)
            return {"found": True, "path": path, "length": len(path) - 1, "expanded": expanded}
        for action in grid.actions(state):
            nxt = grid.transition(state, action)
            if nxt not in parent:
                parent[nxt] = state
                frontier.append(nxt)
    return {"found": False, "path": None, "length": None, "expanded": expanded}


def report(name, grid, result):
    print(f"== {name} ==")
    if not result["found"]:
        print(f"No solution. States expanded: {result['expanded']}\n")
        return
    print(f"Path length: {result['length']}   States expanded: {result['expanded']}")
    print("Path:", " ".join(str(p) for p in result["path"]))
    print(grid.render(result["path"]))
    print()

In [2]:
# Test maps (S = start, G = goal, # = wall)
MAPS_TEXT = {
    "warehouse": """\
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################""",
    "trivial": """\
#####
#SG##
#####""",
    "no_solution": """\
#######
#S....#
###.###
#...#G#
#######""",
    "two_paths": """\
###########
#S........#
#.#######.#
#.#.....#.#
#.#.###.#.#
#.......#G#
###########""",
    "open_room": """\
######################
#S...................#
#....................#
#.......######.......#
#............#.......#
#............#.......#
#............#......G#
######################""",
}

for name, text in MAPS_TEXT.items():
    print(name)
    print(text, end="\n\n")

warehouse
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################

trivial
#####
#SG##
#####

no_solution
#######
#S....#
###.###
#...#G#
#######

two_paths
###########
#S........#
#.#######.#
#.#.....#.#
#.#.###.#.#
#.......#G#
###########

open_room
######################
#S...................#
#....................#
#.......######.......#
#............#.......#
#............#.......#
#............#......G#
######################



### Run on the warehouse map and the two paths map

In [3]:
for m in ["warehouse", "two_paths"]:
    grid = Grid(MAPS_TEXT[m])
    print(f"######## map: {m} ########")
    report("A* (Manhattan)", grid, astar(grid))
    report("BFS", grid, bfs(grid))

######## map: warehouse ########
== A* (Manhattan) ==
Path length: 40   States expanded: 64
Path: (1, 1) (1, 2) (1, 3) (1, 4) (1, 5) (2, 5) (3, 5) (4, 5) (5, 5) (5, 6) (5, 7) (5, 8) (5, 9) (5, 10) (5, 11) (5, 12) (5, 13) (4, 13) (3, 13) (3, 12) (3, 11) (3, 10) (3, 9) (3, 8) (3, 7) (2, 7) (1, 7) (1, 8) (1, 9) (1, 10) (1, 11) (1, 12) (1, 13) (1, 14) (1, 15) (2, 15) (3, 15) (4, 15) (5, 15) (6, 15) (7, 15)
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################

== BFS ==
Path length: 40   States expanded: 64
Path: (1, 1) (1, 2) (1, 3) (1, 4) (1, 5) (2, 5) (3, 5) (4, 5) (5, 5) (5, 6) (5, 7) (5, 8) (5, 9) (5, 10) (5, 11) (5, 12) (5, 13) (4, 13) (3, 13) (3, 12) (3, 11) (3, 10) (3, 9) (3, 8) (3, 7) (2, 7) (1, 7) (1, 8) (1, 9) (1, 10) (1, 11) (1, 12) (1, 13) (1, 14) (1, 15) (2, 15) (3, 15) (4, 15) (5, 15) (6, 15) (7, 15)
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#

## Task 3: Test Results

| Test | Map | Expected | Result |
|---|---|---|---|
| 1 Original warehouse | `warehouse.txt` | some valid path | Found, length 40, 64 states expanded |
| 2 Trivial | `trivial.txt` | 1 step | Found, length 1, 2 expanded |
| 3 No solution | `no_solution.txt` | failure, terminates | "No solution", 9 expanded, no loop |
| 4 Alternative paths | `two_paths.txt` | shortest (12) | Found, length 12 (BFS confirms 12) |

Original warehouse path:

```
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################
```

The unit tests also check that every step is a single orthogonal move into a free cell, and that A* returns the same length as BFS (BFS is optimal for unit costs, so it serves as ground truth).

In [4]:
"""Systematic tests from Task 3.  Run: python -m unittest -v"""
import os
import unittest



def distances_to_goal(grid):
    """Exact h*(n) for every reachable cell: BFS outward from the goal."""
    from collections import deque
    dist = {grid.goal: 0}
    q = deque([grid.goal])
    while q:
        s = q.popleft()
        for a in grid.actions(s):
            n = grid.transition(s, a)
            if n not in dist:
                dist[n] = dist[s] + 1
                q.append(n)
    return dist


def load(name):
    return Grid(MAPS_TEXT[name])


class TestAStar(unittest.TestCase):
    def check_valid_path(self, grid, path):
        self.assertEqual(path[0], grid.start)
        self.assertEqual(path[-1], grid.goal)
        for a, b in zip(path, path[1:]):
            self.assertTrue(grid.passable(b))
            self.assertEqual(abs(a[0] - b[0]) + abs(a[1] - b[1]), 1)

    def test1_original_warehouse(self):
        g = load("warehouse")
        r = astar(g)
        self.assertTrue(r["found"])
        self.check_valid_path(g, r["path"])

    def test2_trivial(self):
        r = astar(load("trivial"))
        self.assertTrue(r["found"])
        self.assertEqual(r["length"], 1)

    def test3_no_solution_terminates(self):
        r = astar(load("no_solution"))
        self.assertFalse(r["found"])
        self.assertIsNone(r["path"])

    def test4_shortest_among_alternatives(self):
        # BFS is optimal for unit costs, so it is the reference answer
        for m in ["warehouse", "two_paths", "open_room"]:
            g = load(m)
            self.assertEqual(astar(g)["length"], bfs(g)["length"], m)

    def test_admissible_heuristics_stay_optimal(self):
        for m in ["warehouse", "two_paths", "open_room"]:
            g = load(m)
            best = bfs(g)["length"]
            for h in (zero, manhattan, euclidean):
                self.assertEqual(astar(g, h)["length"], best, (m, h.__name__))

    def test_manhattan_never_overestimates(self):
        # admissibility check: h(n) <= true distance h*(n) for every reachable cell
        for m in ["warehouse", "two_paths", "open_room"]:
            g = load(m)
            true_dist = distances_to_goal(g)
            for cell, d in true_dist.items():
                self.assertLessEqual(manhattan(cell, g.goal), d)
                self.assertLessEqual(euclidean(cell, g.goal), d)


unittest.main(argv=["notebook"], exit=False, verbosity=2)

test1_original_warehouse (__main__.TestAStar.test1_original_warehouse) ... 

ok


test2_trivial (__main__.TestAStar.test2_trivial) ... 

ok


test3_no_solution_terminates (__main__.TestAStar.test3_no_solution_terminates) ... 

...

test_admissible_heuristics_stay_optimal (__main__.TestAStar.test_admissible_heuristics_stay_optimal) ... 

ok


test_manhattan_never_overestimates (__main__.TestAStar.test_manhattan_never_overestimates) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.015s

OK


## Task 4: Where Each Concept Lives in the Code

| Concept | Location in `search.py` |
|---|---|
| State | `(row, col)` tuples, `Grid.start`, `Grid.goal` |
| Action | `MOVES` dict and `Grid.actions()` |
| Transition | `Grid.transition(state, action)` |
| Goal test | `Grid.is_goal()`, called right after a state is popped |
| g(n) | `g` dict, updated as `new_g = g[state] + 1` |
| h(n) | `heuristic(nxt, goal)` inside the expansion loop |
| f(n) | `key()`: `f_val = g_val + weight * h_val` |
| Frontier | `frontier` list managed with `heapq` |
| Visited states | `closed` set (expanded) plus `g` dict (best known cost) |
| Path reconstruction | `reconstruct(parent, state)` |

(a) A binary heap (`heapq`) used as a priority queue.
(b) `heappop` returns the entry with the smallest f, ties broken by insertion order (or by larger g when `prefer_deeper=True`).
(c) Inside the neighbour loop, when a successor is pushed.
(d) Yes, in `key()`.
(e) A `closed` set skips states already expanded, and a successor is only pushed if its new g beats the best g found so far. Stale heap entries are discarded when popped.

## Task 5: A* vs BFS

| Map | A* found | A* length | A* expanded | BFS found | BFS length | BFS expanded |
|---|---|---|---|---|---|---|
| warehouse | yes | 40 | 64 | yes | 40 | 64 |
| trivial | yes | 1 | 2 | yes | 1 | 2 |
| no_solution | no | n/a | 9 | no | n/a | 9 |
| two_paths | yes | 12 | 23 | yes | 12 | 30 |
| open_room | yes | 24 | 111 | yes | 24 | 111 |

(a) Yes, on every solvable map. (b) Yes, always the same length, as both are optimal here.
(c) On the official warehouse they tie at 64. That map has exactly 64 free cells, and the goal is at the far end of a long winding corridor, so **every** cell must be visited before G regardless of strategy. The heuristic cannot help when the only route goes away from the goal first. On `two_paths`, A* expanded 23 states vs 30 for BFS.
(d) A* expands nodes in order of g + h. Cells with a large estimated remaining distance get pushed back, so effort concentrates on the side of the map that points towards the goal. BFS expands in rings around the start, ignoring where the goal is.

## Task 6: Heuristic Investigation

`two_paths`:

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| h = 0 (uniform cost) | yes | 12 | 30 |
| Manhattan | yes | 12 | 23 |
| Euclidean | yes | 12 | 21 |
| 2 x Manhattan | yes | 12 | 26 |

`open_room` (large open area with one wall):

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| h = 0 (uniform cost) | yes | 24 | 111 |
| Manhattan | yes | 24 | 111 |
| Euclidean | yes | 24 | 111 |
| 2 x Manhattan | yes | **28** | **63** |
| Manhattan, ties broken by larger g | yes | 24 | 68 |

On `warehouse` all variants give length 40 with 64 expanded (see the explanation above).

Observations:

1. **h = 0** turns A* into uniform cost search, which for unit costs behaves like BFS. It is still optimal but uses no goal information, so it expands the most.
2. **Euclidean** is admissible (a straight line is never longer than a 4-connected path) but weaker than Manhattan in theory. Here it expanded slightly fewer states on `two_paths` only because of how ties happened to fall, which shows that small maps can mislead.
3. **2 x Manhattan** is not admissible: it can overestimate h*(n). On `open_room` it returned a path of 28 instead of the optimal 24, but expanded far fewer states. It trades optimality for speed (this is weighted A*).
4. **An unexpected result:** plain Manhattan expanded all 111 states on `open_room`, no better than BFS. The reason is ties. In an open room, a huge number of cells share the same f = 24, and with first-in-first-out tie breaking A* explores all of them. Breaking ties in favour of larger g (nodes closer to the goal) cut this to 68 with the same optimal path. This was found by running the experiment, not by asking the LLM.

**Admissibility:** a unit test computes the true distance h*(n) from every reachable cell (BFS outward from G) and checks Manhattan and Euclidean never exceed it. When h is admissible A* stays optimal. When h overestimates (2x), A* becomes greedier and can commit to a worse path.

### Experiments (reproduces every table in Tasks 3, 5 and 6)

In [5]:
"""
Runs every experiment in Tasks 3, 5 and 6 and prints Markdown tables.
"""
import os

MAPS = ["warehouse", "trivial", "no_solution", "two_paths", "open_room"]


def load(name):
    return Grid(MAPS_TEXT[name])


def fmt(r):
    return ("yes" if r["found"] else "no", r["length"] if r["found"] else "n/a", r["expanded"])


def run_experiments():
    g = load("warehouse")
    free = sum(ch != "#" for row in g.cells for ch in row)
    print(f"Free cells in the official warehouse map: {free}\n")
    print("### Task 3 / Task 5: A* (Manhattan) vs BFS on every map\n")
    print("| Map | A* found | A* length | A* expanded | BFS found | BFS length | BFS expanded |")
    print("|---|---|---|---|---|---|---|")
    for m in MAPS:
        g = load(m)
        a, b = fmt(astar(g)), fmt(bfs(g))
        print(f"| {m} | {a[0]} | {a[1]} | {a[2]} | {b[0]} | {b[1]} | {b[2]} |")

    variants = [
        ("h = 0 (uniform cost)", zero, 1.0),
        ("Manhattan", manhattan, 1.0),
        ("Euclidean", euclidean, 1.0),
        ("2 x Manhattan", manhattan, 2.0),
    ]
    tie_variant = ("Manhattan, ties broken by larger g", manhattan, 1.0)
    for m in ["warehouse", "two_paths", "open_room"]:
        print(f"\n### Task 6: heuristic investigation on `{m}`\n")
        print("| Heuristic | Found | Path length | States expanded |")
        print("|---|---|---|---|")
        g = load(m)
        for name, h, w in variants:
            r = fmt(astar(g, h, w))
            print(f"| {name} | {r[0]} | {r[1]} | {r[2]} |")
        name, h, w = tie_variant
        r = fmt(astar(g, h, w, prefer_deeper=True))
        print(f"| {name} | {r[0]} | {r[1]} | {r[2]} |")


run_experiments()

Free cells in the official warehouse map: 64

### Task 3 / Task 5: A* (Manhattan) vs BFS on every map

| Map | A* found | A* length | A* expanded | BFS found | BFS length | BFS expanded |
|---|---|---|---|---|---|---|
| warehouse | yes | 40 | 64 | yes | 40 | 64 |
| trivial | yes | 1 | 2 | yes | 1 | 2 |
| no_solution | no | n/a | 9 | no | n/a | 9 |
| two_paths | yes | 12 | 23 | yes | 12 | 30 |
| open_room | yes | 24 | 111 | yes | 24 | 111 |

### Task 6: heuristic investigation on `warehouse`

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| h = 0 (uniform cost) | yes | 40 | 64 |
| Manhattan | yes | 40 | 64 |
| Euclidean | yes | 40 | 64 |
| 2 x Manhattan | yes | 40 | 64 |
| Manhattan, ties broken by larger g | yes | 40 | 64 |

### Task 6: heuristic investigation on `two_paths`

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| h = 0 (uniform cost) | yes | 12 | 30 |
| Manhattan | yes | 12 | 23 |
| Euclidean | yes | 12 | 21 |
| 2 x Manhattan

## Task 7: Evaluating the LLM-Generated Agent

| | |
|---|---|
| Designed myself | Problem formulation, state and frontier representation, what to report, all test maps |
| LLM suggested | `heapq` with a counter tie breaker, `closed` set plus `g` dict, the function structure |
| Accepted | The A* and BFS core loops |
| Changed | Added `prefer_deeper` tie breaking after the open room result; rewrote the admissibility test |
| Tested | Five maps, optimality against BFS, admissibility over every cell |

1. The A* and BFS cores were correct immediately.
2. A bug appeared in the first version of the admissibility test: it built a new grid after replacing `S`, which made the map invalid and crashed. The search code itself was fine.
3. Found by running the tests, not by reading the code.
4. The counter tie breaker in the heap tuple was not obvious at first. It exists so Python never has to compare two states when f values are equal.
5. Yes, as listed above.
6. The no solution test (proves termination) and the BFS comparison (proves optimality) were most useful. The open room map revealed the tie breaking behaviour.
7. No. The program printed a believable path on the very first run, but that alone says nothing about optimality or behaviour on unreachable goals.
8. That A* is only as good as the heuristic **and** its tie breaking, and that on a maze where the route leads away from the goal, a heuristic gives no benefit at all.

## Final Reflection

**1. Formulating first.** The formulation decides what counts as a state, which moves are legal and what "best" means. Without it, there is no way to tell if generated code is right, because there is nothing to compare it against. It also exposed details like "cost 1 per move", which is exactly why BFS is a valid optimality check.

**2. Why A* is informed.** It uses h(n), knowledge about the goal that is not part of the problem definition, to decide which frontier node to expand. BFS only uses depth.

**3. Why the heuristic matters.** It controls both efficiency and correctness. h = 0 is safe but slow. An admissible informative h is safe and faster. An overestimating h is fast but can return suboptimal paths, as the 28 vs 24 result showed.

**4. What the LLM contributed.** Fast, clean boilerplate: the heap logic, parent pointers and the reporting format. It turned a written design into running code in minutes.

**5. Risks of not testing.** A plausible path could hide non-optimal results, infinite loops on unreachable goals, or an inadmissible heuristic. In a real warehouse that means wasted energy, missed deadlines, or a robot stuck forever.